**Split pancancer data**

In [23]:
import pandas as pd
import pickle

synlethdb = pd.read_csv('/data/qingyuyang/ESM4SL/data/pancancer/synlethdb_1:1.csv')
genes = set(synlethdb['0']) | set(synlethdb['1'])
# print(type(list(synlethdb['0'])[0]))

with open('/data/qingyuyang/ESM4SL/data/mapping/gid2uid.pkl', 'rb') as f:
    gid2uid = pickle.load(f)
gid_keys = set(gid2uid.keys())
# print(type(list(gid2uid.keys())[0]))

uid2gid = {v: k for k, v in gid2uid.items()}

len(genes), len(gid_keys), len(genes - gid_keys)

(10038, 9845, 193)

In [30]:
with open('/data/qingyuyang/ESM4SL/data/prot_func/genes.txt', 'w') as f:
    f.write(','.join([str(i) for i in list(gid_keys)]))

In [28]:
import numpy as np
import pickle
import os

neg_ratio = 1 # 5
scene = 'CV2' # CV2 

with open(f'/data/qingyuyang/SL_benchmark/data/data_split_wo_comp/{scene}_{neg_ratio}_Exp_indep_test.pkl', 'rb') as f:
    raw_data = pickle.load(f)  # (2, 6, 5)

pos_samples, neg_samples = raw_data
# graph: (9845, 9845); node idx: (num_samples, 2)
graph_train_pos_kfold, graph_val_pos_kfold, graph_test_pos_kfold, train_pos_kfold, val_pos_kfold, test_pos_kfold = pos_samples
graph_train_neg_kfold, graph_val_neg_kfold, graph_test_neg_kfold, train_neg_kfold, val_neg_kfold, test_neg_kfold = neg_samples

out_dir = f'/data/qingyuyang/ESM4SL/data/slbench/NSMexp_indeptest_1:{neg_ratio}/{scene}'
os.makedirs(out_dir, exist_ok=True)

for i in range(5):
    train_pos = train_pos_kfold[i]
    train_neg = train_neg_kfold[i]
    val_pos = val_pos_kfold[i]
    val_neg = val_neg_kfold[i]
    test_pos = test_pos_kfold[i]
    test_neg = test_neg_kfold[i]

    train_pos = np.array([[uid2gid[x[0]], uid2gid[x[1]], 1] for x in train_pos])
    train_neg = np.array([[uid2gid[x[0]], uid2gid[x[1]], 0] for x in train_neg])
    train_data = pd.DataFrame(np.vstack([train_pos, train_neg]), columns=['0', '1', '2'])
    train_data.to_csv(f'{out_dir}/sl_train_{i}.csv', index=False)

    val_pos = np.array([[uid2gid[x[0]], uid2gid[x[1]], 1] for x in val_pos])
    val_neg = np.array([[uid2gid[x[0]], uid2gid[x[1]], 0] for x in val_neg])
    val_data = pd.DataFrame(np.vstack([val_pos, val_neg]), columns=['0', '1', '2'])
    val_data.to_csv(f'{out_dir}/sl_val_{i}.csv', index=False)

    test_pos = np.array([[uid2gid[x[0]], uid2gid[x[1]], 1] for x in test_pos])
    test_neg = np.array([[uid2gid[x[0]], uid2gid[x[1]], 0] for x in test_neg])
    test_data = pd.DataFrame(np.vstack([test_pos, test_neg]), columns=['0', '1', '2'])
    test_data.to_csv(f'{out_dir}/sl_test_{i}.csv', index=False)

/tmp/ipykernel_1971369/3488564294.py:9: DeprecationWarning: Please import `csr_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.csr` namespace is deprecated and will be removed in SciPy 2.0.0.
  raw_data = pickle.load(f)  # (2, 6, 5)


In [13]:
exp_cv1 = np.load('/data/qingyuyang/SL_benchmark/data/data_split_wo_comp/CV1_1_Exp.npy', allow_pickle=True)  # (2, 4, 5)

# print(exp_cv1.shape)
print(exp_cv1[0])
print(exp_cv1[1])

[[<Compressed Sparse Row sparse matrix of dtype 'float64'
  	with 42544 stored elements and shape (9845, 9845)>
  <Compressed Sparse Row sparse matrix of dtype 'float64'
  	with 42546 stored elements and shape (9845, 9845)>
  <Compressed Sparse Row sparse matrix of dtype 'float64'
  	with 42546 stored elements and shape (9845, 9845)>
  <Compressed Sparse Row sparse matrix of dtype 'float64'
  	with 42546 stored elements and shape (9845, 9845)>
  <Compressed Sparse Row sparse matrix of dtype 'float64'
  	with 42546 stored elements and shape (9845, 9845)>    ]
 [<Compressed Sparse Row sparse matrix of dtype 'float64'
  	with 10638 stored elements and shape (9845, 9845)>
  <Compressed Sparse Row sparse matrix of dtype 'float64'
  	with 10636 stored elements and shape (9845, 9845)>
  <Compressed Sparse Row sparse matrix of dtype 'float64'
  	with 10636 stored elements and shape (9845, 9845)>
  <Compressed Sparse Row sparse matrix of dtype 'float64'
  	with 10636 stored elements and shape (

In [6]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import KFold
from utils import numpy2df, get_pairs, are_dataframes_subset, check_C3_property

root_dir = '/data/qingyuyang/ESM4SL/data/slbench/NSMexp_indeptest_1:5/CV1'
# root5_dir = '/data/qingyuyang/ESM4SL/data/slbench/NSMexp_indeptest_1:5/CV1'

train = pd.read_csv(f'{root_dir}/sl_train_0.csv')
val = pd.read_csv(f'{root_dir}/sl_val_0.csv')
test = pd.read_csv(f'{root_dir}/sl_test_0.csv')
all_data = pd.concat([train, val, test], axis=0)

# C3 5-fold split: test genes are unseen in train/val
def cal_ratio(df):
    pos = df[df['2'] == 1].shape[0]
    neg = df[df['2'] == 0].shape[0]
    return pos / neg

C3_save_path = f"{'/'.join(root_dir.split('/')[:-1])}/CV3"
os.makedirs(C3_save_path, exist_ok=True)

genes = np.array(list(set(all_data['0']) | set(all_data['1'])))
sl_pairs = all_data.to_numpy()

index = 0
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_index, test_index in kf.split(genes):
    print(f"Fold {index}:")
    train_genes = genes[train_index]
    test_genes = genes[test_index]

    # split train/valid from train gene
    kf_train_valid = KFold(n_splits=5, shuffle=True, random_state=43)
    for train_train_index, train_valid_index in kf_train_valid.split(train_genes):
        train_train_genes = train_genes[train_train_index]
        train_valid_genes = train_genes[train_valid_index]
        break

    train_data = get_pairs(sl_pairs, train_train_genes, test_genes=None, type=1)
    valid_c3_data = get_pairs(sl_pairs, train_train_genes, test_genes=train_valid_genes, type=3)
    test_c3_data = get_pairs(sl_pairs, train_train_genes, test_genes, type=3)

    train_data = numpy2df(train_data)
    valid_c3_data = numpy2df(valid_c3_data)
    test_c3_data = numpy2df(test_c3_data)
    print(cal_ratio(train_data), cal_ratio(valid_c3_data), cal_ratio(test_c3_data))
    assert are_dataframes_subset(pd.concat([train_data, valid_c3_data, test_c3_data], ignore_index=True), all_data)
    assert check_C3_property(test_c3_data, pd.concat([train_data, valid_c3_data], ignore_index=True))  #
    assert check_C3_property(valid_c3_data, train_data)

    train_data.to_csv(f'{C3_save_path}/sl_train_{index}.csv', index=False)
    valid_c3_data.to_csv(f'{C3_save_path}/sl_val_{index}.csv', index=False)
    test_c3_data.to_csv(f'{C3_save_path}/sl_test_{index}.csv', index=False)
    index += 1

Fold 0:
0.2175114188890526 0.21231593038821955 0.11291942937512557
Fold 1:
0.17878368915303539 0.1697364692547464 0.30783738474434197
Fold 2:
0.21870609848348402 0.1105225988700565 0.20151047030552696
Fold 3:
0.18499484197692956 0.29986859395532195 0.17008681925808997
Fold 4:
0.18691895439133763 0.2751457725947522 0.19747550240823783


**Select positive samples for SL discovery experiments**

In [5]:
import pandas as pd

input_dir = '/data/qingyuyang/ESM4SL/data/slbench/NSMexp_indeptest_1:1/CV1'
cv1_train_0 = pd.read_csv(f'{input_dir}/sl_train_0.csv')
cv1_val_0 = pd.read_csv(f'{input_dir}/sl_val_0.csv')
cv1_test_0 = pd.read_csv(f'{input_dir}/sl_test_0.csv')
all_data = pd.concat([cv1_train_0, cv1_val_0, cv1_test_0], axis=0, ignore_index=True)
all_pos = all_data[all_data['2'] == 1]
# all_pos.to_csv('/data/qingyuyang/ESM4SL/data/slbench/raw_data.csv', index=False)

In [6]:
import pickle

with open('/data/qingyuyang/ESM4SL/data/mapping/name2id.pkl', 'rb') as f:
    gn2gid = pickle.load(f)

gid2gn = {int(gid): gn for gn, gid in gn2gid.items()}

In [7]:
all_pos['0'] = all_pos['0'].map(gid2gn)
all_pos['1'] = all_pos['1'].map(gid2gn)

all_pos.rename({'0': 'gene1', '1': 'gene2', '2': 'label'}, axis=1, inplace=True)

all_pos.to_csv('/data/qingyuyang/ESM4SL/data/slbench/raw_data.csv', index=False)

/tmp/ipykernel_493148/649791921.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_pos['0'] = all_pos['0'].map(gid2gn)
/tmp/ipykernel_493148/649791921.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_pos['1'] = all_pos['1'].map(gid2gn)
/tmp/ipykernel_493148/649791921.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-

In [2]:
import torch

a = torch.load('/data/qingyuyang/ESM4SL/data/protrek_text_emb/2.pt')
a.shape

/tmp/ipykernel_177487/3721823394.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  a = torch.load('/data/qingyuyang/ESM4SL/data/protrek_text_emb/2.pt')


torch.Size([22, 1024])

In [2]:
import pickle

with open('/data/qingyuyang/ESM4SL/data/mapping/name2id.pkl', 'rb') as f:
    name2id = pickle.load(f)

In [3]:
import pandas as pd

logit = pd.read_csv('/data/qingyuyang/ESM4SL/output/pfnew/A549/AKT1/fold_0/csv_log/version_0/test_logits.csv')
logit.sort_values('probs', ascending=False)

,gene1,gene2,probs
8,207,10110,9.991032e-01
79,207,7428,9.957609e-01
3,207,8986,9.847925e-01
0,207,10000,9.639756e-01
6,207,6199,9.510011e-01
...,...,...,...
54,207,2956,1.138149e-06
72,207,6608,5.475866e-07
29,207,1029,3.542874e-07
77,207,7298,1.136313e-07


In [4]:
import pickle

with open('/data/qingyuyang/ESM4SL/data/mapping/gid2uid.pkl', 'rb') as f:
    gid2uid = pickle.load(f)
uid2gid = {v: k for k, v in gid2uid.items()}

In [ ]:
case_dir = '/data/qingyuyang/ESM4SL/data/SLKB/slbench/case/A549/AKT1'
methods = ['KG4SL', 'NSF4SL', 'PTGNN', 'SLMGAE']
for m in methods:
    test_logit = pd.read_csv(f'{case_dir}/{m}/sl_test_0.csv')

In [9]:
import pickle

with open('/data/qingyuyang/SL/ESM4SL/data/SLKB/all_id_seq.pkl', 'rb') as f:
    id_seq = dict(pickle.load(f))

genes = set(id_seq.keys())

In [10]:
import pandas as pd

slkb = pd.read_csv('/data/qingyuyang/SL/ESM4SL/data/SLKB/SLKB_processed.csv')
slkb_genes=set(slkb['0']) | set(slkb['1'])

In [11]:
slkb_genes==genes

True

In [2]:
import pandas as pd
import os
train = pd.read_csv('/data/qingyuyang/SL/ESM4SL/data/SLKB/specific/MELJUSO/Tail/sl_train_0.csv')
val = pd.read_csv('/data/qingyuyang/SL/ESM4SL/data/SLKB/specific/MELJUSO/Tail/sl_val_0.csv')
genes = list(set(train['0'])|set(train['1'])|set(val['0'])|set(val['1']))

esm_emb_genes = [int(i.split('.')[0]) for i in os.listdir('/data/qingyuyang/SL/ESM4SL/data/SLKB/ESM2_emb')]

In [3]:
set(genes).difference(set(esm_emb_genes))

set()

In [1]:
import torch
temp = torch.randn((2, 3))
print(temp.shape)
a = torch.cat([temp], dim=1)
print(a.shape)

torch.Size([2, 3])
torch.Size([2, 3])


In [3]:
a[2:]

tensor([], size=(0, 3))

In [6]:
import os
import numpy as np
from tqdm import tqdm

esm_root = '/data/qingyuyang/SL/ESM4SL/data/SLKB/ESM2_emb'

for fn in tqdm(os.listdir(esm_root)):
    gid = fn.split('.')[0]
    gene = np.load(os.path.join(esm_root, f"{gid}.npy"), allow_pickle=True).item()['rep']
    gene = torch.from_numpy(gene)
    torch.save(gene, f'{esm_root}/{gid}.pt')
    # break

100%|██████████| 5868/5868 [01:48<00:00, 54.12it/s] 


In [8]:
a = torch.load('/data/qingyuyang/SL/ESM4SL/data/SLKB/ESM2_emb/9.pt')
a.shape

/tmp/ipykernel_2417361/2454044586.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  a = torch.load('/data/qingyuyang/SL/ESM4SL/data/SLKB/ESM2_emb/9.pt')


torch.Size([290, 1280])

In [16]:
sorted(list(set(range(24)).difference(set([]))))


[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23]

In [ ]:
import pickle

with open('/data/qingyuyang/SL/MuSL/processed_data/all_emb_scgpt.pkl', 'rb') as f:
    emb_scgpt = pickle.load(f)

In [5]:
len(emb_scgpt), list(emb_scgpt.values())[0].shape

(7684, (512,))

In [ ]:
import pickle

with open('/data/qingyuyang/SL/SLformer/data/saved_data/map/gene2ensembl.pkl', 'rb') as f:
    a = pickle.load(f)

a

{'TSPAN6': 'ENSG00000000003',
 'TNMD': 'ENSG00000000005',
 'DPM1': 'ENSG00000000419',
 'SCYL3': 'ENSG00000000457',
 'C1orf112': 'ENSG00000000460',
 'FGR': 'ENSG00000000938',
 'CFH': 'ENSG00000000971',
 'FUCA2': 'ENSG00000001036',
 'GCLC': 'ENSG00000001084',
 'NFYA': 'ENSG00000001167',
 'STPG1': 'ENSG00000001460',
 'NIPAL3': 'ENSG00000001461',
 'LAS1L': 'ENSG00000001497',
 'ENPP4': 'ENSG00000001561',
 'SEMA3F': 'ENSG00000001617',
 'CFTR': 'ENSG00000001626',
 'ANKIB1': 'ENSG00000001629',
 'CYP51A1': 'ENSG00000001630',
 'KRIT1': 'ENSG00000001631',
 'RAD52': 'ENSG00000002016',
 'MYH16': 'ENSG00000002079',
 'BAD': 'ENSG00000002330',
 'LAP3': 'ENSG00000002549',
 'CD99': 'ENSGR0000002586',
 'HS3ST1': 'ENSG00000002587',
 'AOC1': 'ENSG00000002726',
 'WNT16': 'ENSG00000002745',
 'HECW1': 'ENSG00000002746',
 'MAD1L1': 'ENSG00000002822',
 'LASP1': 'ENSG00000002834',
 'SNX11': 'ENSG00000002919',
 'TMEM176A': 'ENSG00000002933',
 'M6PR': 'ENSG00000003056',
 'KLHL13': 'ENSG00000003096',
 'CYP26B1': 'E

In [2]:
len(a)

71948

In [4]:
import pickle

with open('/data/qingyuyang/SL/PFSL/data/mapping/name2id.pkl', 'rb') as f:
    a = pickle.load(f)

a

{'PHYHIP': '9796',
 'GPANK1': '7918',
 'ZRSR2': '8233',
 'NRF1': '4899',
 'PI4KA': '5297',
 'SLC15A1': '6564',
 'EIF3I': '8668',
 'FAXDC2': '10826',
 'MT1A': '4489',
 'SORT1': '6272',
 'LRP5': '4041',
 'RPS8': '6202',
 'CD7': '924',
 'MRPL9': '65005',
 'PSMC4': '5704',
 'EGR1': '1958',
 'SNRPD2': '6633',
 'KRTAP4-5': '85289',
 'TAF1': '6872',
 'VAV3': '10451',
 'KRTAP5-9': '3846',
 'HMOX2': '3163',
 'FOS': '2353',
 'DNAJC3': '5611',
 'LGALS9': '3965',
 'MYC': '4609',
 'MAP3K6': '9064',
 'KRT20': '54474',
 'TUBGCP5': '114791',
 'F2': '2147',
 'DNM1': '1759',
 'NUDC': '10726',
 'CD53': '963',
 'GSK3B': '2932',
 'NAXD': '55739',
 'SLC7A14': '57709',
 'PVR': '5817',
 'CLCN7': '1186',
 'CCNB1': '891',
 'ATF5': '22809',
 'CDK2': '1017',
 'NDUFS8': '4728',
 'PITX1': '5307',
 'PIK3R1': '5295',
 'SRPK1': '6732',
 'HOXD12': '3238',
 'BZW2': '28969',
 'TAB1': '10454',
 'CDKL4': '344387',
 'PPP1R16A': '84988',
 'ETS2': '2114',
 'SP1': '6667',
 'NKIRAS2': '28511',
 'IFIT2': '3433',
 'AURKA': '6790'

In [5]:
len(a)

13459

In [2]:
with open('/data/qingyuyang/SL/MVGCNiSL/data/cellline_feats/22RV1_cnv.txt', 'r') as f:
    cnv = dict([l.split() for l in f.readlines()])

cnv = {key: float(value) for key, value in cnv.items()}
cnv

{'DDX11L1': 0.9759788689762816,
 'WASH7P': 0.9759788689762816,
 'OR4F5': 0.9759788689762816,
 'DDX11L17': 0.9759788689762816,
 'FAM87B': 0.9759788689762816,
 'LINC00115': 0.9759788689762816,
 'LINC01128': 0.9759788689762816,
 'FAM41C': 0.9759788689762816,
 'SAMD11': 0.9759788689762816,
 'NOC2L': 0.9759788689762816,
 'KLHL17': 0.9759788689762816,
 'PLEKHN1': 0.9759788689762816,
 'PERM1': 0.9759788689762816,
 'HES4': 0.9759788689762816,
 'ISG15': 0.9759788689762816,
 'AGRN': 0.9759788689762816,
 'RNF223': 0.9759788689762816,
 'C1orf159': 0.9759788689762816,
 'LINC01342': 0.9759788689762816,
 'MIR200B': 0.9759788689762816,
 'MIR200A': 0.9759788689762816,
 'MIR429': 0.9759788689762816,
 'TTLL10': 0.9759788689762816,
 'TNFRSF18': 0.9759788689762816,
 'TNFRSF4': 0.9759788689762816,
 'SDF4': 0.9759788689762816,
 'B3GALT6': 0.9759788689762816,
 'C1QTNF12': 0.9759788689762816,
 'UBE2J2': 0.9759788689762816,
 'SCNN1D': 0.9759788689762816,
 'ACAP3': 0.9759788689762816,
 'MIR6726': 0.9759788689762

In [9]:
import torch

a = torch.load('/data/qingyuyang/SL/PFSL/data/pretrained_emb/esmc/9.pt') #, weights_only=False
a.shape

torch.Size([290, 2560])

In [8]:
import os
from tqdm import tqdm

root = '/data/qingyuyang/SL/PFSL/data/pretrained_emb/esmc'
for fn in tqdm(os.listdir(root)):
    complete_fn = f'{root}/{fn}'
    embedding = torch.load(complete_fn)[0]

    L = embedding.shape[0]
    embedding = embedding[1:L-1, :]
    torch.save(embedding, complete_fn)

100%|██████████| 3016/3016 [01:41<00:00, 29.69it/s] 


In [1]:
import pandas as pd

a = pd.read_csv('/data/qingyuyang/SL/PFSL/data/SLKB/specific/22RV1/C1/sl_test_0.csv')[['0', '1', '2']]
a.to_csv('/data/qingyuyang/SL/PFSL/data/SLKB/specific/22RV1/C1/sl_test_0.csv', index=False)